# TCC - Análise Preditiva de Risco Acadêmico Escolar
## CRISP-DM: 2. Data Understanding


## Objetivos da fase

- Listar quais arquivos CSV alimentam a pipeline.
- Distinguir bases centrais da pipeline analítica e bases de apoio para relatórios.
- Entender a granularidade de cada arquivo.
- Verificar quantidade de linhas, colunas e campos ausentes.
- Observar pequenas amostras públicas com registros artificiais consistentes.
- Conferir cobertura de anos letivos, séries, turmas, disciplinas e professores.
- Identificar problemas que precisam ser tratados na fase seguinte, Data Preparation.


## Carregar bibliotecas

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


## Localizar os CSVs canônicos

A pipeline pública do projeto não depende diretamente do banco institucional. Ela espera arquivos CSV já extraídos em `artifacts/database/csv/`, conforme o contrato público documentado em `docs/ENTRADA_DE_DADOS_E_CONTRATOS.md`.



In [ ]:
def discover_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "artifacts" / "database" / "csv").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = discover_project_root()
CSV_DIR = PROJECT_ROOT / "artifacts" / "database" / "csv"

EXPECTED_FILES = {
    "alunos": "aluno.csv",
    "notas": "media_nota_aluno.csv",
    "faltas": "faltas_aluno.csv",
    "pagamentos": "pagamento_aluno.csv",
    "responsaveis": "responsaveis_aluno.csv",
    "professores": "professor_disciplina.csv",
}

CSV_DIR

## Carregar arquivos disponíveis

Se algum CSV não existir localmente, ele aparece como ausente no inventário. Isso permite que o notebook seja aberto também por avaliadores que não tenham acesso aos insumos locais originais.


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    return pd.read_csv(path, encoding="utf-8", low_memory=False)


tables = {
    name: read_csv_if_exists(CSV_DIR / filename)
    for name, filename in EXPECTED_FILES.items()
}

inventory = []
for name, filename in EXPECTED_FILES.items():
    frame = tables[name]
    inventory.append({
        "base": name,
        "arquivo": filename,
        "existe": frame is not None,
        "linhas": 0 if frame is None else len(frame),
        "colunas": 0 if frame is None else len(frame.columns),
    })

pd.DataFrame(inventory)

### Visualmente, o que saiu desta etapa

A tabela acima é a primeira fotografia real do dado dentro do notebook. Ela mostra, de forma objetiva, quais CSVs existem, o tamanho de cada base e quais arquivos ainda estariam faltando para o restante da pipeline.


### Exemplo visual fixo do inventário

Abaixo está um exemplo estático do que a célula acima entrega quando o dado é carregado:

<div style="font-size: 1em; overflow-x: auto;">

| base | arquivo | existe | linhas | colunas |
| --- | --- | --- | --- | --- |
| alunos | aluno.csv | True | 19661 | 26 |
| notas | media_nota_aluno.csv | True | 239678 | 20 |
| faltas | faltas_aluno.csv | True | 168051 | 22 |
| pagamentos | pagamento_aluno.csv | True | 220090 | 26 |
| professores | professor_disciplina.csv | True | 1624 | 15 |
| responsáveis | responsaveis_aluno.csv | True | 39256 | 26 |

</div>


In [ ]:
inventory_frame = pd.DataFrame(inventory).sort_values("base").reset_index(drop=True)
inventory_frame

Aqui o dado ainda não foi transformado. Ele apenas foi localizado e carregado. O ganho desta etapa é enxergar a disponibilidade real das bases antes de qualquer análise mais profunda.



### O que aconteceu com os dados após esta célula

Quando você executar a célula acima, ela não altera nenhum CSV. Ela apenas monta um inventário seguro dizendo quais arquivos existem, quantas linhas possuem e quantas colunas carregam.

A leitura didática aqui é simples:
- se um arquivo central estiver ausente, as fases seguintes ficam comprometidas;
- se um arquivo opcional ou de apoio estiver ausente, o notebook ainda abre, mas a leitura correspondente fica limitada;
- diferenças grandes de volume entre bases já sugerem, desde cedo, que a fase seguinte precisará tratar granularidades diferentes.



## Dicionário conceitual dos arquivos

<div style="font-size: 1em; overflow-x: auto;">

| Base | Arquivo | Papel na pipeline | Granularidade real/esperada |
|---|---|---|---|
| Alunos | `aluno.csv` | contexto escolar e matrícula; apoio para conferência documental | aluno x matrícula x turma x período |
| Notas | `media_nota_aluno.csv` | histórico principal de desempenho e origem do alvo de próxima nota | aluno x período x turma x disciplina x etapa |
| Faltas | `faltas_aluno.csv` | sinal complementar de frequência para a análise escolar | evento de falta por aluno x período x disciplina x etapa |
| Pagamentos | `pagamento_aluno.csv` | contexto financeiro do aluno no período letivo | movimento financeiro por aluno x matrícula x período |
| Responsáveis | `responsaveis_aluno.csv` | apoio cadastral e para relatórios; não entra no núcleo analítico atual | aluno x matrícula x responsável |
| Professores | `professor_disciplina.csv` | vínculo pedagógico por turma e disciplina; arquivo opcional quando esse vínculo não vier na extração | período x turma x disciplina x professor |

</div>

Na arquitetura atual, as bases `notas`, `faltas`, `pagamentos` e `professores` concentram os sinais centrais da pipeline analítica. As bases `alunos` e `responsáveis` funcionam principalmente como apoio contextual, documental e para relatórios. Nesta fase, o objetivo é verificar se esses arquivos existem, qual o papel de cada um e se possuem as chaves mínimas para seguir adiante com segurança.




## Funções para amostra pública

A base pública deste projeto já trabalha com registros artificiais consistentes com o contrato publicado. Por isso, as amostras abaixo podem usar os nomes do próprio CSV, como `Aluno 00418` e `Professor 00207`, sem precisar aplicar uma segunda transformação no notebook.



### Exemplo fixo de nomes artificiais nas amostras

Nomes exibidos no notebook devem ser lidos como rótulos artificiais consistentes com a versão pública, mantendo apenas a forma necessária para entender as bases.


In [ ]:
def public_preview(frame: pd.DataFrame, rows: int = 5, max_columns: int = 12) -> pd.DataFrame:
    preview = frame.head(rows).copy()
    if len(preview.columns) > max_columns:
        preview = preview.iloc[:, :max_columns]
    return preview.fillna("<ausente>")


for name, frame in tables.items():
    if frame is not None:
        print(f"\n{name} - amostra pública")
        display(public_preview(frame))


### Pequena amostra visual do que chegou em cada base

Assim como no notebook de referência, a saída acima deixa claro como os dados "parecem" depois da leitura. Como esta versão do projeto já trabalha com CSVs públicos baseados em registros artificiais, o notebook pode mostrar os nomes da própria base sem precisar trocá-los por hashes artificiais.

In [ ]:
preview_catalog = []
for name, frame in tables.items():
    if frame is None or frame.empty:
        continue
    preview = public_preview(frame, rows=2, max_columns=6)
    preview_catalog.append({
        "base": name,
        "linhas_mostradas": len(preview),
        "colunas_mostradas": ", ".join(preview.columns.astype(str).tolist()),
    })

pd.DataFrame(preview_catalog)

Esse quadro-resumo funciona como legenda da amostra: ele diz exatamente qual recorte de cada base foi exibido logo acima.

### O que observar nas amostras com nomes artificiais

A célula acima mostra a forma dos dados sem expor pessoas reais. O objetivo não é inspecionar nomes ou IDs, mas enxergar o desenho da base: quais colunas aparecem juntas, que tipos de valores existem e se a granularidade bate com o contrato esperado.

Quando a amostra estiver correta, você deve conseguir reconhecer o papel da base sem precisar ver nenhum dado sensível.



## Colunas disponíveis por base

Este bloco ajuda a verificar se os CSVs seguem o contrato esperado. Ele mostra apenas nomes de colunas, não registros institucionais originais.


### Exemplo visual fixo das colunas principais

A tabela abaixo resume as colunas mais importantes para entender o papel de cada CSV. As bases centrais da pipeline analítica são `notas`, `faltas`, `pagamentos` e `professores`; `alunos` e `responsáveis` aparecem como apoio contextual e documental.

<div style="font-size: 1em; overflow-x: auto;">

| base | colunas_chave |
| --- | --- |
| alunos | IDAluno, IDMatricula, NomePeriodo, NomeSerie, IDTurma, SituacaoMatricula |
| notas | IDAluno, NomePeriodo, NomeDisciplina, NomeEtapa, ValorMedia, IDTurma, IDDisciplina |
| faltas | IDAluno, NomePeriodo, NomeDisciplina, NomeEtapa, DataFalta |
| pagamentos | IDAluno, NomePeriodo, IDMovimento, ValorMovimento, PagoMovimento, EhMensalidadeMovimento, DataVencimentoMovimento, DataAntecipadoMovimento |
| responsáveis | IDAluno, IDResponsavel, TipoResponsavel, NomeResponsavel |
| professores | IDUnidade, IDPeriodo, IDCurso, IDSerie, IDTurma, IDDisciplina, IDFuncionario, NomeFuncionario |

</div>


In [ ]:
columns_summary = []
for name, frame in tables.items():
    if frame is None:
        columns_summary.append({"base": name, "colunas": "arquivo ausente"})
    else:
        columns_summary.append({"base": name, "colunas": ", ".join(frame.columns)})

pd.DataFrame(columns_summary)

### O que aconteceu com o dado após listar as colunas

Depois desta célula, o notebook deixa de mostrar apenas exemplos de valores e passa a mostrar a estrutura formal de cada CSV. Isso ajuda a verificar se o contrato esperado foi realmente respeitado.


In [ ]:
column_count_frame = pd.DataFrame([
    {
        "base": name,
        "quantidade_colunas": 0 if frame is None else len(frame.columns),
        "primeiras_colunas": "arquivo ausente" if frame is None else ", ".join(frame.columns[:8].astype(str).tolist()),
    }
    for name, frame in tables.items()
])
column_count_frame

### O que esta saída confirma

Aqui você enxerga o dicionário real que chegou do ambiente local. Se alguma coluna esperada não aparecer, o problema ainda não é de preparação nem de modelagem: ele já nasceu na extração ou no contrato do CSV.

Esse bloco é importante porque separa duas perguntas diferentes:
- o dado existe?
- o dado existe no formato que a pipeline espera?




## Qualidade inicial: valores ausentes

Valores ausentes são verificados para identificar problemas reais de cobertura nas bases que alimentam a pipeline. Nesta leitura, campos totalmente vazios ou cadastrais fora do escopo analítico atual são ocultados para não gerar dúvidas na apresentação.

Assim, as tabelas abaixo mostram apenas ausências relevantes para a compreensão da qualidade inicial dos dados.




In [ ]:
MISSING_TABLE_TITLES = {
    "alunos": "alunos - maiores percentuais de ausência",
    "notas": "notas - maiores percentuais de ausência",
    "faltas": "faltas - maiores percentuais de ausência",
    "pagamentos": "pagamentos - maiores percentuais de ausência",
    "responsaveis": "responsáveis - maiores percentuais de ausência",
    "professores": "professores - maiores percentuais de ausência",
}


def is_out_of_scope_missing_column(column: object) -> bool:
    normalized = str(column).lower()
    return "cep" in normalized or ("nascimento" in normalized and "responsavel" in normalized)


def missing_report(frame: pd.DataFrame, top: int = 10) -> pd.DataFrame:
    missing = frame.isna().sum().sort_values(ascending=False)
    report = missing.to_frame("ausentes")
    report["percentual"] = (report["ausentes"] / len(frame) * 100).round(2)
    report = report.reset_index(names="coluna")
    report = report[report["ausentes"] > 0]
    report = report[report["ausentes"] < len(frame)]
    report = report[~report["coluna"].apply(is_out_of_scope_missing_column)]
    return report.head(top).reset_index(drop=True)


for name, frame in tables.items():
    if frame is not None and len(frame) > 0:
        display(Markdown(f"#### {MISSING_TABLE_TITLES.get(name, name)}"))
        report = missing_report(frame)
        if report.empty:
            report = pd.DataFrame([{
                "coluna": "sem ausências relevantes",
                "ausentes": 0,
                "percentual": 0.0,
            }])
        display(report)



### Exemplo visual fixo dos percentuais de ausentes

Os quadros abaixo mostram os maiores percentuais de ausência relevantes em cada base. Campos totalmente vazios e campos cadastrais fora do escopo analítico atual foram retirados da exibição para evitar a leitura equivocada de que seriam variáveis usadas pela pipeline.

#### alunos

<div style="font-size: 1em; overflow-x: auto;">

| coluna | ausentes | percentual |
| --- | ---: | ---: |
| NomeCurso | 985 | 5.01% |
| NomeSerie | 985 | 5.01% |
| DataMatricula | 86 | 0.44% |

</div>

#### notas

<div style="font-size: 1em; overflow-x: auto;">

| coluna | ausentes | percentual |
| --- | ---: | ---: |
| ValorMedia | 3405 | 1.42% |

</div>

#### faltas

<div style="font-size: 1em; overflow-x: auto;">

| coluna | ausentes | percentual |
| --- | ---: | ---: |
| sem ausências relevantes | 0 | 0.00% |

</div>

#### pagamentos

<div style="font-size: 1em; overflow-x: auto;">

| coluna | ausentes | percentual |
| --- | ---: | ---: |
| ValorPagoMovimento | 26949 | 12.24% |
| NomeCurso | 6502 | 2.95% |
| NomeSerie | 6502 | 2.95% |

</div>

#### responsáveis

<div style="font-size: 1em; overflow-x: auto;">

| coluna | ausentes | percentual |
| --- | ---: | ---: |
| NomeCurso | 1951 | 4.97% |
| NomeSerie | 1951 | 4.97% |
| SexoResponsavel | 2 | 0.01% |

</div>

#### professores

<div style="font-size: 1em; overflow-x: auto;">

| coluna | ausentes | percentual |
| --- | ---: | ---: |
| sem ausências relevantes | 0 | 0.00% |

</div>




### Pequena saída visual do efeito dos ausentes

A listagem acima mostra onde cada base chega mais incompleta, ignorando campos totalmente vazios ou cadastrais que não entram na leitura analítica atual. Para deixar isso ainda mais visível, o quadro abaixo resume o maior percentual relevante de ausência encontrado em cada CSV.



In [ ]:
missing_summary = []
for name, frame in tables.items():
    if frame is None or frame.empty:
        continue
    report = missing_report(frame, top=1)
    if report.empty:
        continue
    top_row = report.iloc[0]
    missing_summary.append({
        "base": name,
        "coluna_mais_ausente": top_row["coluna"],
        "ausentes": int(top_row["ausentes"]),
        "percentual": float(top_row["percentual"]),
    })

pd.DataFrame(missing_summary).sort_values("percentual", ascending=False)



## Cobertura temporal e acadêmica

Como o projeto faz previsão temporal, é essencial entender quais anos letivos existem e como eles se distribuem entre as bases. Nesta fase, o foco ainda não é separar conjuntos experimentais, mas confirmar se a cobertura temporal é suficiente para sustentar a leitura histórica do problema depois.



### Exemplo visual fixo da cobertura acadêmica

Em vez de repetir linhas parecidas, a tabela abaixo resume melhor o alcance temporal de cada base:

<div style="font-size: 1em; overflow-x: auto;">

| base | ano_inicial | ano_final | anos_distintos | disciplinas_distintas |
| --- | --- | --- | --- | --- |
| notas | 2020 | 2026 | 7 | 70 |
| faltas | 2023 | 2026 | 4 | 57 |
| pagamentos | 2004 | 2026 | 23 | <não se aplica> |
| professores | 2022 | 2026 | 5 | 66 |

</div>


In [ ]:
coverage_rows = []
for name, frame in tables.items():
    if frame is None:
        continue
    row = {"base": name, "linhas": len(frame)}
    if "NomePeriodo" in frame.columns:
        periods = sorted(frame["NomePeriodo"].dropna().astype(str).unique().tolist())
        row["periodos_exemplo"] = ", ".join(periods[:6])
    for column in ["NomePeriodo", "NomeCurso", "NomeSerie", "ApelidoTurma", "NomeDisciplina"]:
        if column in frame.columns:
            row[f"{column}_distintos"] = frame[column].nunique(dropna=True)
    coverage_rows.append(row)

pd.DataFrame(coverage_rows).fillna("-")

### O que a cobertura temporal parece visualmente

A tabela acima resume quantos períodos, séries, turmas e disciplinas aparecem em cada base. O quadro abaixo foca ainda mais no aspecto temporal, mostrando diretamente quais anos letivos estão presentes em cada CSV.

In [ ]:
period_coverage = []
for name, frame in tables.items():
    if frame is None or "NomePeriodo" not in frame.columns:
        continue
    values = sorted(frame["NomePeriodo"].dropna().astype(str).unique().tolist())
    period_coverage.append({
        "base": name,
        "periodos_detectados": ", ".join(values[:10]),
        "quantidade_periodos": len(values),
    })

pd.DataFrame(period_coverage)

### O que a cobertura temporal revela

Depois desta célula, você passa a saber se existe base suficiente para leitura temporal do problema. Se os períodos estiverem muito concentrados em um único ano, as fases seguintes ficarão limitadas porque a evolução histórica do aluno quase não poderá ser observada.

Também é aqui que aparece uma visão rápida da cobertura acadêmica: quantos cursos, séries, turmas e disciplinas estão realmente representados em cada CSV.




## Entendimento da base de notas

A base de notas é a principal fonte do problema analítico. Nesta fase, o interesse é entender sua granularidade, sua cobertura e se ela realmente registra a sequência de desempenho escolar que o trabalho pretende estudar. Aqui também importa confirmar se `NomePeriodo` e `NomeEtapa` chegam em formato legível, porque a pipeline depende dessa leitura para ordenar os registros no tempo.



In [ ]:
grades = tables.get("notas")
if grades is not None and "ValorMedia" in grades.columns:
    grade_values = pd.to_numeric(grades["ValorMedia"], errors="coerce")
    display(pd.DataFrame({
        "metrica": ["linhas", "alunos", "disciplinas", "menor_nota", "media", "maior_nota"],
        "valor": [
            len(grades),
            grades["IDAluno"].nunique() if "IDAluno" in grades.columns else None,
            grades["NomeDisciplina"].nunique() if "NomeDisciplina" in grades.columns else None,
            grade_values.min(),
            grade_values.mean(),
            grade_values.max(),
        ],
    }))

    if "NomePeriodo" in grades.columns:
        display(grades.groupby("NomePeriodo", dropna=False).size().reset_index(name="registros_de_nota"))

### Como a base de notas fica visível após a leitura

Uma pequena amostra estatística e uma distribuição concreta por período. Isso ajuda a perceber se o dado de notas chega com volume e continuidade suficientes para sustentar a análise histórica.


### Exemplo visual fixo da base de notas

Abaixo está uma pequena amostra estática, filtrada para anos recentes e já usando os nomes artificiais da base pública:

<div style="font-size: 1em; overflow-x: auto;">

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeAluno | NomeDisciplina | NomeEtapa | ValorMedia |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 1º BIMESTRE | 10.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 2º BIMESTRE | 10.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 3º BIMESTRE | 10.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 4º BIMESTRE | 7.9 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 1º BIMESTRE | 8.1 |

</div>


In [ ]:
grade_sample = public_preview(
    grades,
    rows=5,
    max_columns=8,
)

grade_sample

A amostra acima mostra como uma linha típica da base de notas se apresenta depois da leitura, já pronta para ser entendida na fase seguinte.

### O que a base de notas mostra após a leitura

A base de notas passa a revelar o coração do problema. Quando você executa a célula acima, você deixa de ver apenas colunas e passa a enxergar a distribuição real do histórico escolar: quantos registros existem, quantos alunos aparecem e como isso se espalha pelos períodos.

Se essa base estiver fraca ou mal distribuída, toda a análise do trabalho fica comprometida, porque ela é o registro mais direto da evolução acadêmica do aluno.



## Entendimento da base de faltas

As faltas entram como sinal complementar. Nesta fase, o objetivo é entender como esses eventos chegam no CSV, qual sua granularidade e se o volume observado faz sentido para apoiar a leitura acadêmica.

In [ ]:
absences = tables.get("faltas")
if absences is not None:
    summary = {
        "eventos_de_falta": len(absences),
        "alunos_com_falta": absences["IDAluno"].nunique() if "IDAluno" in absences.columns else None,
        "disciplinas_com_falta": absences["NomeDisciplina"].nunique() if "NomeDisciplina" in absences.columns else None,
    }
    display(pd.DataFrame([summary]))

    group_cols = [column for column in ["NomePeriodo", "NomeSerie"] if column in absences.columns]
    if group_cols:
        display(absences.groupby(group_cols, dropna=False).size().reset_index(name="faltas").head(20))

### Como a base de faltas aparece visualmente

A saída acima resume o volume observado. A amostra abaixo mostra como um evento de falta aparece no dado bruto antes de qualquer tratamento posterior.

### Exemplo visual fixo da base de faltas

Abaixo está uma pequena amostra estática, filtrada para anos recentes e já usando os nomes artificiais da base pública:

<div style="font-size: 1em; overflow-x: auto;">

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeAluno | NomeDisciplina | NomeEtapa | DataFalta |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Arte | 3º BIMESTRE | 2025-08-14 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 2º BIMESTRE | 2025-05-09 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 2º BIMESTRE | 2025-05-13 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 2º BIMESTRE | 2025-05-23 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Ciências da Natureza | 3º BIMESTRE | 2025-08-12 |

</div>


In [ ]:
if absences is not None and not absences.empty:
    absence_sample = public_preview(absences, rows=5, max_columns=8)
    display(absence_sample)

### O que muda quando as faltas entram na leitura

A partir desta saída, faltas deixam de ser apenas eventos soltos e passam a ser percebidas como um sinal escolar com valor analítico.

O que se observa aqui é se existe massa crítica para acompanhar frequência por aluno, período, disciplina e etapa sem perder o sentido temporal do registro.




## Entendimento da base de pagamentos

Os pagamentos não determinam sozinhos o desempenho acadêmico, mas podem funcionar como contexto. Nesta fase, o objetivo é entender se a base financeira possui volume, colunas e continuidade suficientes para apoiar a leitura do aluno sem precisar expor detalhes individualizados. Também é importante observar se campos como `PagoMovimento`, `EhMensalidadeMovimento`, `DataVencimentoMovimento` e `DataAntecipadoMovimento` chegam de forma consistente, porque essa base costuma ser a mais heterogênea entre os CSVs de entrada.




In [ ]:
payments = tables.get("pagamentos")
if payments is not None:
    summary = {
        "movimentos_financeiros": len(payments),
        "alunos_com_movimento": payments["IDAluno"].nunique() if "IDAluno" in payments.columns else None,
        "coluna_status_pagamento": "PagoMovimento" in payments.columns,
        "coluna_mensalidade": "EhMensalidadeMovimento" in payments.columns,
    }
    display(pd.DataFrame([summary]))

    if "PagoMovimento" in payments.columns:
        display(payments["PagoMovimento"].astype(str).value_counts(dropna=False).reset_index(name="quantidade"))

### Como a base de pagamentos aparece visualmente

Depois da contagem de status, a amostra abaixo mostra o formato de uma linha financeira real. Isso ajuda a entender que o dado ainda está em nível transacional nesta fase.

### Exemplo visual fixo da base de pagamentos

Abaixo está uma pequena amostra estática, filtrada para anos recentes e já usando os nomes artificiais da base pública:

<div style="font-size: 0.9em; overflow-x: auto;">

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeAluno | DescricaoMovimento | PagoMovimento | EhMensalidadeMovimento | ValorMovimento | ValorPagoMovimento |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | 6o ano - Lights, camera, action - 2o SEMESTRE | True | False | 65.0 | 65.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | 6o ano - 3o bim - A coragem de ser imperfeito | True | False | 45.0 | 45.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Kit Integrado 6º ano EF | True | False | 2820.0 | 2600.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | Agenda 2025 | True | False | 45.0 | 45.0 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Aluno 00418 | 6o ano - 4o bim - Tempos e Jeitos para viver | True | False | 45.0 | 45.0 |

</div>


In [ ]:
if payments is not None and not payments.empty:
    payment_sample = public_preview(payments, rows=5, max_columns=8)
    display(payment_sample)

### O que a leitura financeira acrescenta

Depois desta célula, a base de pagamentos deixa de parecer apenas administrativa e ganha papel analítico. Ela não define sozinha o desempenho escolar, mas passa a funcionar como contexto para o risco.

Didaticamente, esta etapa responde: existe sinal financeiro suficiente para ser considerado no trabalho, ou a base está incompleta demais para sustentar essa leitura?



## Entendimento da base de professores

A base de professores permite entender se o projeto conseguirá relacionar disciplina, turma e docente. Nesta fase, o foco é verificar se esses vínculos existem com cobertura suficiente, sem ainda discutir como serão consolidados depois. Como esse CSV é opcional na arquitetura atual, o notebook também precisa registrar quando ele estiver ausente para que a limitação fique documentada desde cedo.




In [ ]:
teachers = tables.get("professores")
if teachers is not None:
    summary = {
        "vinculos_professor_disciplina": len(teachers),
        "professores_distintos": teachers["IDFuncionario"].nunique() if "IDFuncionario" in teachers.columns else None,
        "disciplinas_com_professor": teachers["NomeDisciplina"].nunique() if "NomeDisciplina" in teachers.columns else None,
        "turmas_com_professor": teachers["IDTurma"].nunique() if "IDTurma" in teachers.columns else None,
    }
    display(pd.DataFrame([summary]))

### Como a base de professores aparece visualmente

O resumo numérico acima mostra quantos vínculos existem. A amostra abaixo mostra o formato de um vínculo pedagógico logo na leitura do CSV.

### Exemplo visual fixo da base de professores

Abaixo está uma pequena amostra estática, filtrada para anos recentes e já usando os nomes artificiais da base pública:

<div style="font-size: 1em; overflow-x: auto;">

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeDisciplina | NomeFuncionario |
| --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Arte | Professor 00207 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Ciências da Natureza | Professor 00071 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Educação Física | Professor 00205 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Filosofia | Professor 00091 |
| 2025 | Ensino Fundamental | 6º Ano | 6º Ano A - Matutino | Geografia | Professor 00167 |

</div>


In [ ]:
if teachers is not None and not teachers.empty:
    teacher_sample = public_preview(teachers, rows=5, max_columns=8)
    display(teacher_sample)

### Como ler a base de professores

Esta saída mostra se o projeto conseguirá explicar o contexto pedagógico por docente, turma e disciplina. Se houver poucos vínculos ou muitos vazios, essa limitação precisa ser registrada antes da preparação.

Isso é importante porque evita descartar turmas inteiras só porque o banco não informou o professor em todos os casos.



## Verificacao das chaves de integração

A leitura integrada das bases depende principalmente destas chaves e colunas operacionais:

- `IDAluno`
- `NomePeriodo`
- `NomeDisciplina`
- `NomeEtapa`
- leitura confiável de `NomeEtapa` para gerar a ordem temporal (`stage_order`)
- `IDUnidade`, `IDPeriodo`, `IDCurso`, `IDSerie`, `IDTurma`, `IDDisciplina` para professor

`aluno.csv` e `responsaveis_aluno.csv` permanecem importantes para contexto e relatórios, mas não são a espinha dorsal da integração analítica atual.

O objetivo desta verificação é encontrar colunas ausentes antes da preparação dos dados.



### Exemplo visual fixo da verificação das chaves

<div style="font-size: 1em; overflow-x: auto;">

| ligacao | chave | objetivo |
| --- | --- | --- |
| notas + faltas | IDAluno + NomePeriodo + NomeDisciplina + stage_order | verificar se as bases podem ser alinhadas por etapa |
| notas + pagamentos | IDAluno + NomePeriodo | verificar se o contexto financeiro pode ser relacionado ao período letivo |
| notas + professores | IDUnidade + IDPeriodo + IDCurso + IDSerie + IDTurma + IDDisciplina | verificar se existe vínculo pedagógico utilizável sem misturar turmas ou séries diferentes |

</div>


In [ ]:
required_columns = {
    "notas": ["IDAluno", "NomePeriodo", "NomeDisciplina", "NomeEtapa", "ValorMedia"],
    "faltas": ["IDAluno", "NomePeriodo", "NomeDisciplina", "NomeEtapa", "DataFalta"],
    "pagamentos": ["IDAluno", "NomePeriodo", "PagoMovimento", "EhMensalidadeMovimento", "ValorMovimento", "ValorPagoMovimento", "DataAntecipadoMovimento", "DataVencimentoMovimento"],
    "professores": ["IDUnidade", "IDPeriodo", "IDCurso", "IDSerie", "IDTurma", "IDDisciplina", "IDFuncionario", "NomeFuncionario"],
}

checks = []
for name, required in required_columns.items():
    frame = tables.get(name)
    available = set() if frame is None else set(frame.columns)
    for column in required:
        checks.append({
            "base": name,
            "coluna": column,
            "presente": column in available,
        })

pd.DataFrame(checks)

### Saída final desta fase de entendimento

Depois desta verificação, o notebook entrega uma resposta visual simples: quais colunas-chave existem e quais faltam. O quadro abaixo resume essa leitura por base.

In [ ]:
check_summary = (
    pd.DataFrame(checks)
    .groupby("base", dropna=False)["presente"]
    .agg(total_colunas_verificadas="size", colunas_presentes="sum")
    .reset_index()
)
check_summary["colunas_ausentes"] = check_summary["total_colunas_verificadas"] - check_summary["colunas_presentes"]
check_summary

Quando este bloco fecha, a fase de Data Understanding já mostrou não só o código de leitura, mas também o aspecto visual do dado e o estado em que cada base chegou para a pipeline.

### O que esta verificação entrega

Ao final desta checagem, você sabe se as bases centrais possuem as chaves mínimas para serem unidas sem improviso. Se uma coluna obrigatória estiver faltando, o problema precisa ser resolvido antes da preparação.

Em outras palavras: aqui termina a fase em que ainda estamos entendendo o dado. A partir daqui, a próxima etapa já passa a transformá-lo.


## Primeiras conclusões desta fase

Ao final do Data Understanding, espera-se responder:

- Os CSVs mínimos existem?
- Ha anos letivos suficientes para sustentar leitura temporal do problema?
- As bases centrais possuem as chaves e colunas necessárias para integração?
- Existem campos ausentes que precisam de tratamento?
- Ha turmas, disciplinas ou professores com cobertura incompleta?
- A base de notas possui sequências suficientes para acompanhar a evolução entre etapas?

Essas respostas orientam a fase seguinte, Data Preparation, onde as bases passarão a ser normalizadas, integradas e transformadas.


